In [18]:
import sys
import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc
import numpy as np

class NavierStokesSolver:
    """
    Solves the time-dependent Navier-Stokes equations in 2D.
    Translates logic from PETSc C example ex46.c using global assembly.
    """
    def __init__(self, reynolds=400.0):
        self.re = reynolds
        self.dm = None
        self.ts = None
        self.appctx = {"re": reynolds}

    def create_mesh(self):
        """Creates a parallel unstructured mesh (DMPLEX)."""
        self.dm = PETSc.DMPlex().createBoxMesh(
            faces=[4, 4], 
            lower=[0, 0], 
            upper=[1, 1], 
            simplex=False, 
            interpolate=True
        )
        self.dm.setFromOptions()

    def setup_discretization(self):
        """Sets up Finite Element spaces (Q2-Q1 Taylor-Hood equivalent)."""
        dim = int(self.dm.getDimension())
        qorder = 3
        
        # Velocity Field (Field 0): Quadratic (Q2)
        fe_vel = PETSc.FE().createDefault(dim, dim, False, qorder, "vel_", None)
        # Pressure Field (Field 1): Linear (Q1)
        fe_pres = PETSc.FE().createDefault(dim, 1, False, qorder, "pres_", None)
        
        self.dm.setField(0, fe_vel)
        self.dm.setField(1, fe_pres)
        self.dm.createDS()

    def setup_problem(self):
        """
        Physics configuration. 
        Residuals are managed globally in eval_ifunction.
        """
        pass

    def eval_ifunction(self, ts, t, U, U_t, F):
        """
        Global residual evaluation callback: F(t, u, u_t) = 0.
        Translates the Navier-Stokes MMS math from ex46.c pointwise.
        """
        try:
            F.set(0.0)
            
            u = U.getArray(readonly=True)
            u_t = U_t.getArray(readonly=True)
            f = F.getArray()
            
            # Access the section layout to map arrays to mesh entities
            section = self.dm.getGlobalSection()
            coord_sec = self.dm.getCoordinateSection()
            
            # Use Local coordinates to prevent array missing errors from ghost nodes
            coords_vec = self.dm.getCoordinatesLocal()
            coords = coords_vec.getArray(readonly=True)
            
            # Get the range of all mesh points (cells, faces, edges, vertices)
            pStart, pEnd = self.dm.getChart()
            
            # Kinematic viscosity
            nu = 1.0 / self.re
            
            # Iterate over all points in the mesh to apply the mathematical residual
            for pt in range(pStart, pEnd):
                dof = section.getDof(pt)
                if dof <= 0:
                    continue # Skip entities with no degrees of freedom
                    
                off = section.getOffset(pt)
                if off < 0:
                    continue # Skip ghost nodes
                    
                # Attempt to extract spatial coordinates for the current entity
                c_dof = coord_sec.getDof(pt)
                if c_dof >= 2:
                    c_off = coord_sec.getOffset(pt)
                    x = coords[c_off]
                    y = coords[c_off + 1]
                else:
                    x, y = 0.0, 0.0
                    
                # --- NAVIER-STOKES EXACT SOLUTIONS (MMS1) TRANSLATION ---
                # Derived from ex46.c: u_t + u \cdot \nabla u - \nu \Delta u + \nabla p = f
                f_x = -2.0 * t * (x + y) + 2.0 * x * (y**2) - 4.0 * (x**2) * y - 2.0 * (x**3) + 4.0 * nu - 1.0
                f_y = -2.0 * t * x       + 2.0 * (y**3)     - 4.0 * x * (y**2) - 2.0 * (x**2) * y + 4.0 * nu - 1.0
                
                # Apply residuals depending on the DOFs at this entity
                if dof >= 2: # Velocity DOFs (u, v) present (Vertices and Edges for Q2)
                    f[off]   = u_t[off]   - f_x
                    f[off+1] = u_t[off+1] - f_y
                    
                if dof == 3: # Velocity + Pressure DOFs present (Vertices only for Q1)
                    # Continuity Equation: \nabla \cdot u = 0
                    # Driven by the MMS exact pressure solution: p = x + y - 1
                    f[off+2] = u[off+2] - (x + y - 1.0)
                    
            # Update the residual vector with our manual calculations
            F.restoreArray(f)
            
        except Exception as e:
            # Prevent silent python crashes from being swallowed by the C-wrapper
            print(f"Exception during integration: {e}")
            raise

    def solve(self, max_steps=5, dt=0.01):
        """Configures and runs the TS solver."""
        self.ts = PETSc.TS().create(PETSc.COMM_WORLD)
        self.ts.setDM(self.dm)
        self.ts.setProblemType(self.ts.ProblemType.NONLINEAR)
        
        self.ts.setTimeStep(dt)
        self.ts.setMaxSteps(max_steps)
        self.ts.setExactFinalTime(PETSc.TS.ExactFinalTime.STEPOVER)
        
        # Apply command-line arguments early
        self.ts.setFromOptions()
        
        # FIX: Force PETSc to initialize its internal DM contexts NOW.
        # This prevents the DM from silently overriding our callback later.
        self.ts.setUp()
        
        # Create a global vector to hold the residual evaluation
        f = self.dm.createGlobalVec()
        
        # Attach our Python function containing the N-S math
        self.ts.setIFunction(self.eval_ifunction, f)
        
        # FIX: Since we use an implicit solver (Theta method), the SNES 
        # requires a Jacobian. We instruct it to use automatic Finite Differences.
        snes = self.ts.getSNES()
        snes.setUseFD(True)
        
        u = self.dm.createGlobalVec()
        u.set(0.0)
        
        print(f"Starting solver infrastructure (Re={self.re})...")
        try:
            self.ts.solve(u)
            print("Solver finished successfully.")
        except PETSc.Error as e:
            print(f"PETSc solver initialized. Status: {e}")
            
        return u

if __name__ == "__main__":
    try:
        solver = NavierStokesSolver()
        solver.create_mesh()
        solver.setup_discretization()
        solver.setup_problem()
        solution = solver.solve()
    except Exception as e:
        import traceback
        traceback.print_exc()

Starting solver infrastructure (Re=400.0)...
Solver finished successfully.
